In [70]:
import numpy as np
from tmu.models.classification.vanilla_classifier import TMClassifier
import pandas as pd
from sklearn.model_selection import train_test_split
import argparse
from datetime import datetime
import random
from sklearn.utils import shuffle
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
import numpy as np
from statistics import mean, stdev
from scipy.stats import linregress
from tmu.preprocessing.standard_binarizer.binarizer import StandardBinarizer

In [71]:
def tm_classifier(
    number_of_clauses=30,
    T=22,
    s=3.0,
    platform="CPU",
    weighted_clauses=True,
    feature_negation=True
):
    return TMClassifier(
        number_of_clauses=number_of_clauses,
        T=T,
        s=s,
        platform=platform,
        weighted_clauses=weighted_clauses,
        feature_negation=feature_negation
    )

In [115]:
def process_csv_to_dict(csv_file, test_size=0.2, random_state=42):

    # Load the CSV file
    df = pd.read_csv(csv_file)
    columns_to_drop = ['filename','channel','segment','label_str','label']
    # Separate features and target variable
    cols_to_drop = [col for col in df.columns if col.endswith(('min', 'max', 'median'))]
    columns = cols_to_drop + columns_to_drop
    X = df.drop(columns=columns)  # Features
    Y = df['label']  # Labels
    

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=test_size, random_state=random_state)
    print( f'shape {X_train.shape} , {X_test.shape} ')

    # binarize with current max_bits
    binarizer = StandardBinarizer(max_bits_per_feature=3)  # cxhange binarization bits if required
    binarizer.fit(X_train.to_numpy())
    data_dict = {
        "x_train": binarizer.transform(X_train.to_numpy()).astype(np.uint32),
        "x_test": binarizer.transform(X_test.to_numpy()).astype(np.uint32),
        "y_train": y_train.to_numpy().astype(np.uint32),
        "y_test": y_test.to_numpy().astype(np.uint32),
    }
    
    return data_dict

In [116]:
channel =1     # sensor number
mfcc_dict = process_csv_to_dict(f"time_to_frequency/mfcc/channel_{channel}_ms.csv")   #process proper file path fro each sensor

shape (3984, 12) , (996, 12) 


In [104]:
data = mfcc_dict
x_train, y_train = data['x_train'], data['y_train']
x_test, y_test = data['x_test'], data['y_test']

In [105]:
np.unique(y_train)

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint32)

In [106]:
tm = tm_classifier(number_of_clauses=30,T=22,s=3)
for i in range(100):
        batch_data_log = {}
        tm.fit(x_train, y_train)
        tm_test_pred , tm_test_scores = tm.predict(x_test , return_class_sums=True)
        acc = 100 * (tm_test_pred == y_test).mean()
        if (i%10 == 0 ):
            print(f'epoch - {i} : accuracy = {acc}')

epoch - 0 : accuracy = 40.36144578313253
epoch - 10 : accuracy = 60.040160642570285
epoch - 20 : accuracy = 58.734939759036145
epoch - 30 : accuracy = 62.14859437751004
epoch - 40 : accuracy = 64.05622489959839
epoch - 50 : accuracy = 60.94377510040161
epoch - 60 : accuracy = 63.955823293172685
epoch - 70 : accuracy = 62.75100401606426
epoch - 80 : accuracy = 65.86345381526104
epoch - 90 : accuracy = 60.94377510040161


In [109]:
import numpy as np

num_classes = 10
output_file = f"sensor_{channel}_clause_information.txt"

with open(output_file, "w") as f:
    for the_class in range(num_classes):

        # Get weights and literals
        weights = tm.weight_banks[the_class].get_weights()   #weights can be extracted for each class 
        literals = tm.clause_banks[the_class].get_literals()  #literal include and exclude status can be extracted for each class

        f.write(f"{'='*60}\n")
        f.write(f"Class {the_class}\n")
        f.write(f"{'='*60}\n\n")

        # Write weights
        f.write("Clause Weights (30 clauses):\n")
        for i, w in enumerate(weights):
            f.write(f"  Clause {i:02d}: {w}\n")

        f.write("\n")

        # Write literals
        f.write("Clause Literals (Include=1, Exclude=0, shape=30x24):\n")
        for clause_idx in range(literals.shape[0]):
            f.write(f"  Clause {clause_idx:02d}: ")
            literal_str = " ".join(map(str, literals[clause_idx]))
            f.write(literal_str + "\n")

        f.write("\n\n")

print(f"Saved TM clause information to '{output_file}'")

Saved TM clause information to 'sensor_7_clause_information.txt'


In [114]:
tm_test_pred , tm_test_scores = tm.predict(x_test , return_class_sums=True)
acc = 100 * (tm_test_pred == y_test).mean()
print(f'Test Accuracy = {acc}')

Test Accuracy = 64.2570281124498


In [113]:
num_classes = 10
num_clauses = 30
num_literals = 24
half = num_literals // 2

output_file = f"sensor_{channel}_rules.txt"

with open(output_file, "w") as f:
    for cls in range(num_classes):

        literals = tm.clause_banks[cls].get_literals()
        weights = tm.weight_banks[cls].get_weights()

        f.write("=" * 80 + "\n")
        f.write(f"Class {cls} Rules\n")
        f.write("=" * 80 + "\n")

        for clause_idx in range(num_clauses):

            clause_literals = literals[clause_idx]
            weight = weights[clause_idx]

            rule_terms = []

            for lit_idx, val in enumerate(clause_literals):
                if val == 1:
                    if lit_idx < half:
                        rule_terms.append(f"x{lit_idx}")
                    else:
                        rule_terms.append(f"NOT x{lit_idx - half}")

            # Handle empty clause
            if not rule_terms:
                rule_str = "TRUE"
            else:
                rule_str = " AND ".join(rule_terms)

            f.write(
                f"Rule-{clause_idx + 1:02d} (weight={weight}): {rule_str}\n"
            )

        f.write("\n\n")

print(f"Rules saved to '{output_file}'")

Rules saved to 'sensor_7_rules.txt'
